In [1]:
from pathlib import Path
import xarray as xr
import torch
import torch.nn.functional
from climanet.st_encoder_decoder import SpatioTemporalModel
from climanet.utils import pred_to_numpy, set_seed, configure_compute_resources, plot_results, plot_histograms
from climanet.train import train_monthly_model
from climanet.predict import predict_monthly_var
from climanet import STDataset

from torch.utils.data import random_split

import time

## Read data

In [2]:
# without diurnal cycle
data_folder = Path("./eso4clima/watervapor")
run_dir = "./runs_hourly_watervapor"
var_name = "prw"

file_names = [data_folder / f"202001_hr_ERA5_masked_{var_name}.nc", data_folder / f"202002_hr_ERA5_masked_{var_name}.nc"]
hourly_data = xr.open_mfdataset(file_names, chunks={"time": 1, "latitude": 160, "longitude": 160}).sortby("lat")

file_names = [data_folder / f"202001_mon_ERA5_full_{var_name}.nc", data_folder / f"202002_mon_ERA5_full_{var_name}.nc"]
monthly_data = xr.open_mfdataset(file_names, chunks={"time": 1, "latitude": 160, "longitude": 160}).sortby("lat")

file_name = data_folder / "era5_lsm_bool.nc"  # downloded from era5 and regridded using the function `regrid_to_boundary_centered_grid`
lsm_mask = xr.open_dataset(file_name)
# lsm_mask = lsm_mask.rename({'latitude': 'lat', 'longitude': 'lon'})

## Subset data (for fast example)

In [3]:
# # coordinates of subset
# lon_subset = slice(-50, 50) 
# lat_subset = slice(-30, 10)

# tiny sample
lon_subset = slice(-10, 10) 
lat_subset = slice(-10, 10)

daily_subset = hourly_data.sel(lon=lon_subset, lat=lat_subset)
monthly_subset = monthly_data.sel(lon=lon_subset, lat=lat_subset)
lsm_subset = lsm_mask.sel(lon=lon_subset, lat=lat_subset)  # True=Land

print(daily_subset[var_name].shape, monthly_subset[var_name].shape)  # (time, lat, lon)

(1440, 40, 40) (2, 40, 40)


In [4]:
# calculate residuals as target
daily_subset_averaged = daily_subset.resample(time="MS").mean(skipna=True)
daily_subset_averaged["time"] = monthly_subset["time"]

# Residuals
monthly_subset_res = monthly_subset - daily_subset_averaged

## Train the model

In [5]:
# create the model (small)
set_seed()

patch_size = (1, 4, 4)
model = SpatioTemporalModel(patch_size=patch_size, overlap=2, embed_dim=64, dropout=0.2, hidden=64)

In [6]:
# create dataset
num_patches = (10, 10)
spatial_patch_size = (patch_size[1]*num_patches[0], patch_size[2]*num_patches[1])
stride = (spatial_patch_size[0] // 5, spatial_patch_size[1] // 5)

dataset = STDataset(
    input_da=daily_subset[var_name],
    monthly_da=monthly_subset_res[var_name],
    land_mask=lsm_subset["lsm"],
    patch_size=(1, *spatial_patch_size),  # based on the patch_size in model
    stride=stride,
    sh_embed_dim=96,
    sh_order_L = 10,
    is_hourly=True,
    
)
print(len(dataset))

Patch grid (m x i x j): 2 x 1 x 1 = 2 patches
Overlap: 32 pixels (height), 32 pixels (width)
2


In [7]:
# create train test data
generator = torch.Generator().manual_seed(42)
train_size = int(0.6 * len(dataset))
validation_size = int(0.3 * len(dataset))
test_size = len(dataset) - train_size - validation_size
train_dataset, validation_dataset, test_dataset = random_split(dataset, [train_size, validation_size, test_size], generator=generator)
print(len(train_dataset), len(validation_dataset), len(test_dataset))

1 0 1


### Start training loop

In [8]:
# Device and resources
compute_threads = 120
dataloader_num_workers = 4
device = "cpu"
model = configure_compute_resources(model, device=device, compute_threads=compute_threads, dataloader_num_workers=dataloader_num_workers)

In [9]:
start = time.time()
# verbose is True
batch_size = 200
trained_model = train_monthly_model(
    model, train_dataset, validation_dataset=validation_dataset, batch_size=batch_size, num_epoch=10, accumulation_steps=20, device=device, run_dir=run_dir, dataloader_num_workers=dataloader_num_workers,
)

total_time = (time.time() - start) / 60.0
print(f"{total_time:.2f}")

AttributeError: 'float' object has no attribute 'item'

## Inspect results and compare

In [ ]:
# inference on test data, verbose is True
batch_size = 200
predictions = predict_monthly_var(
    model=f"{run_dir}/best_model.pth", dataset=test_dataset, batch_size=batch_size, save_predictions=True, device=device, run_dir=run_dir, dataloader_num_workers=dataloader_num_workers,
)

In [ ]:
# inference on whole dataset
num_patches = (10, 10)
spatial_patch_size = (patch_size[1]*num_patches[0], patch_size[2]*num_patches[1])

dataset = STDataset(
    input_da=daily_subset[var_name],
    monthly_da=monthly_subset_res[var_name],
    land_mask=lsm_subset["lsm"],
    patch_size=(1, *spatial_patch_size),  # based on the patch_size in model
    sh_embed_dim=96,
    sh_order_L = 10,
    is_hourly=True,    
)
print(len(dataset))

In [ ]:
batch_size = 200
predictions_res = predict_monthly_var(
    model=f"{run_dir}/best_model.pth", dataset=dataset, batch_size=batch_size, verbose=False, save_predictions=True, device=device, run_dir=run_dir, dataloader_num_workers=dataloader_num_workers,
)

In [ ]:
# add to monthly average
predictions = daily_subset_averaged[var_name] + predictions_res["predictions"]

In [ ]:
ocean = ~lsm_subset["lsm"].values
masked_target = monthly_subset[var_name].where(ocean)
masked_pred = predictions.where(ocean)
plot_results(masked_target, masked_pred, label="SST K", title=("Target", "Prediction"))

In [ ]:
err_baseline = (monthly_subset[var_name] - daily_subset_averaged[var_name])  # M - mean(D)
err_predictions = (monthly_subset[var_name] - predictions)  # M - M_hat

ocean = ~lsm_subset["lsm"].values
masked_err_baseline = err_baseline.where(ocean) 
masked_err_predictions = err_predictions.where(ocean) 

plot_results(masked_err_baseline, masked_err_predictions, label="error K", title=("err_Baseline", "err_Prediction"), error=True)

In [ ]:
plot_histograms(masked_err_baseline, masked_err_predictions, label="error K", legend_labels=("err_Baseline", "err_Prediction"))